# Aegis Threat Detection - Equal Class Balancing & Dataset Preparation
This notebook creates a **perfectly balanced dataset** by sampling an equal number of packets for each of the 6 categories (`BENIGN`, `DoS`, `PortScan`, `Infiltration`, `Bot`, `Brute Force`) across all three source datasets: **CICIDS2017**, **UNSW-NB15**, and **Synthetic Threats**.

### Pipeline Steps:
1. Ingest all split files from `data/train/` and `data/val/` to compile a complete pool of samples.
2. Standardize and align labels.
3. Determine class support counts and sample an equal number of records per class.
4. Perform a stratified 60-40 split on the balanced set.
5. Export `balanced_train.csv` and `balanced_val.csv`.

In [1]:
import os
import gc
import random
import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)

## 1. Define Paths and Constants

In [2]:
DATA_DIR = 'data'
TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VAL_DIR = os.path.join(DATA_DIR, 'val')

# Source split files we generated earlier
source_files = [
    os.path.join(TRAIN_DIR, 'cicids_train.csv'),
    os.path.join(VAL_DIR, 'cicids_val.csv'),
    os.path.join(TRAIN_DIR, 'unsw_train.csv'),
    os.path.join(VAL_DIR, 'unsw_val.csv'),
    os.path.join(TRAIN_DIR, 'synthetic_train.csv'),
    os.path.join(VAL_DIR, 'synthetic_val.csv')
]

OUTPUT_TRAIN = os.path.join(DATA_DIR, 'balanced_train.csv')
OUTPUT_VAL = os.path.join(DATA_DIR, 'balanced_val.csv')

print("Source files for pooling:")
for f in source_files:
    print(f"  - {f} (Exists: {os.path.exists(f)})")

Source files for pooling:
  - data\train\cicids_train.csv (Exists: True)
  - data\val\cicids_val.csv (Exists: True)
  - data\train\unsw_train.csv (Exists: True)
  - data\val\unsw_val.csv (Exists: True)
  - data\train\synthetic_train.csv (Exists: True)
  - data\val\synthetic_val.csv (Exists: True)


## 2. Ingest and Pool Records Memory-Efficiently
Since the overall datasets combined are several gigabytes, we load them in chunks and isolate the rows by their labels into dictionary lists to minimize RAM overhead.

In [3]:
class_pools = {
    'BENIGN': [],
    'DoS': [],
    'PortScan': [],
    'Infiltration': [],
    'Bot': [],
    'Brute Force': []
}

dtypes = {f'payload_byte_{i}': np.uint8 for i in range(1, 1501)}
dtypes['ttl'] = np.uint16
dtypes['total_len'] = np.uint32
dtypes['protocol'] = str
dtypes['t_delta'] = np.float32
dtypes['label'] = str

chunksize = 100000

for f_path in source_files:
    if not os.path.exists(f_path):
        continue
    print(f"Pooling from {f_path}...")
    for chunk in pd.read_csv(f_path, dtype=dtypes, chunksize=chunksize):
        for label, group in chunk.groupby('label'):
            if label in class_pools:
                class_pools[label].append(group)
                
print("Consolidating dataframes for each class...")
for label in class_pools:
    if len(class_pools[label]) > 0:
        class_pools[label] = pd.concat(class_pools[label], ignore_index=True)
        print(f"  Class '{label}': {len(class_pools[label])} total pooled rows.")
    else:
        class_pools[label] = pd.DataFrame(columns=chunk.columns)
        print(f"  Class '{label}': 0 pooled rows.")

gc.collect()

Pooling from data\train\cicids_train.csv...
Pooling from data\val\cicids_val.csv...
Pooling from data\train\unsw_train.csv...
Pooling from data\val\unsw_val.csv...
Pooling from data\train\synthetic_train.csv...
Pooling from data\val\synthetic_val.csv...
Consolidating dataframes for each class...
  Class 'BENIGN': 388108 total pooled rows.
  Class 'DoS': 849563 total pooled rows.
  Class 'PortScan': 13392 total pooled rows.
  Class 'Infiltration': 198467 total pooled rows.
  Class 'Bot': 7636 total pooled rows.
  Class 'Brute Force': 107970 total pooled rows.


0

## 3. Balance Classes (Equal Sampling)
We determine the maximum possible size for equal classes (governed by the smallest class size), sample them, and concatenate into a unified dataframe.

In [4]:
# Determine the minimum support size
min_samples = min(len(class_pools[lbl]) for lbl in class_pools)
print(f"\nSmallest class has {min_samples} samples.")
print(f"Sampling exactly {min_samples} rows from each class to construct a balanced dataset...")

balanced_dfs = []
for lbl in class_pools:
    sampled_df = class_pools[lbl].sample(n=min_samples, random_state=42)
    balanced_dfs.append(sampled_df)
    
master_balanced_df = pd.concat(balanced_dfs, ignore_index=True)
print(f"Master balanced dataset created. Shape: {master_balanced_df.shape}")
print(master_balanced_df['label'].value_counts())

# Clear pools to free RAM
del class_pools, balanced_dfs
gc.collect()


Smallest class has 7636 samples.
Sampling exactly 7636 rows from each class to construct a balanced dataset...
Master balanced dataset created. Shape: (45816, 1505)
label
BENIGN          7636
DoS             7636
PortScan        7636
Infiltration    7636
Bot             7636
Brute Force     7636
Name: count, dtype: int64


0

## 4. Stratified Train/Val Split (60-40) and Export
We split the master balanced set into 60% training and 40% validation sets, keeping classes balanced in both.

In [5]:
train_dfs = []
val_dfs = []

for label, group in master_balanced_df.groupby('label'):
    # Shuffle group
    group = group.sample(frac=1, random_state=42).reset_index(drop=True)
    split_point = int(len(group) * 0.6)
    
    train_dfs.append(group.iloc[:split_point])
    val_dfs.append(group.iloc[split_point:])
    
train_df = pd.concat(train_dfs, ignore_index=True)
val_df = pd.concat(val_dfs, ignore_index=True)

print(f"Final train set shape: {train_df.shape}")
print(train_df['label'].value_counts())

print(f"\nFinal validation set shape: {val_df.shape}")
print(val_df['label'].value_counts())

# Export to CSV
print(f"\nSaving balanced datasets...")
train_df.to_csv(OUTPUT_TRAIN, index=False)
val_df.to_csv(OUTPUT_VAL, index=False)

print(f"  Saved train set to: {OUTPUT_TRAIN}")
print(f"  Saved val set to: {OUTPUT_VAL}")

print("\nDataset preparation complete!")

Final train set shape: (27486, 1505)
label
BENIGN          4581
Bot             4581
Brute Force     4581
DoS             4581
Infiltration    4581
PortScan        4581
Name: count, dtype: int64

Final validation set shape: (18330, 1505)
label
BENIGN          3055
Bot             3055
Brute Force     3055
DoS             3055
Infiltration    3055
PortScan        3055
Name: count, dtype: int64

Saving balanced datasets...
  Saved train set to: data\balanced_train.csv
  Saved val set to: data\balanced_val.csv

Dataset preparation complete!
